"""
=============================================================================
PANEL MAESTRO — SCHOOL DROPOUT PREDICTION COLOMBIA
=============================================================================

Output:
    Data/Processed/panel_maestro.parquet
    Data/Processed/panel_maestro.csv
    Data/Processed/diagnostico_panel.xlsx

Primary key:
    SEDE_CODIGO x PERIODO_ANIO

Period:
    2018-2023

Train:
    2018-2022

Test:
    2023

Excluded source:
    Terridata

Python:
    3.10+

=============================================================================
"""

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
import unicodedata
import re

warnings.filterwarnings("ignore")


# =============================================================================
# 0. PATHS & CONFIG
# =============================================================================

ROOT = Path(
    r"C:\Users\DELL\OneDrive\Escritorio\UNIVERSIDAD\Maestria Business A\Proyecto Empresarial\Data"
)

RAW = ROOT / "Raw"
PROCESSED = ROOT / "Processed"

PROCESSED.mkdir(parents=True, exist_ok=True)


# -------------------------------------------------------------------------
# PANEL PERIOD
# -------------------------------------------------------------------------

YEARS = [2018, 2019, 2020, 2021, 2022, 2023]

TRAIN_YEARS = [2018, 2019, 2020, 2021, 2022]
TEST_YEAR = 2023


# -------------------------------------------------------------------------
# C-600 FILES
# -------------------------------------------------------------------------

C600_FILES = {
    "jornadas": "Jornadas_nivel",
    "desplazados": "Desplazados",
    "limitacion": "Limitacion_fisica",
    "tradicional": "Ed_tradicional",
    "flexible": "Ed_Flexible",
    "etnia": "Etnia",
}


# -------------------------------------------------------------------------
# IPM REGION MAPPING
# -------------------------------------------------------------------------

REGION_MAP = {
    1: "Caribe",
    2: "Oriental",
    3: "Central",
    4: "Bogota",
    5: "Antioquia",
    6: "Valle",
    7: "Pacifica",
    8: "Orinoquia",
    9: "San_Andres",
}


DPTO_TO_REGION = {
    "08": "Caribe",
    "13": "Caribe",
    "20": "Caribe",
    "23": "Caribe",
    "44": "Caribe",
    "47": "Caribe",
    "70": "Caribe",
    "15": "Oriental",
    "25": "Oriental",
    "54": "Oriental",
    "68": "Oriental",
    "17": "Central",
    "18": "Central",
    "41": "Central",
    "63": "Central",
    "66": "Central",
    "73": "Central",
    "11": "Bogota",
    "05": "Antioquia",
    "76": "Valle",
    "19": "Pacifica",
    "27": "Pacifica",
    "52": "Pacifica",
    "81": "Orinoquia",
    "85": "Orinoquia",
    "86": "Orinoquia",
    "91": "Orinoquia",
    "94": "Orinoquia",
    "95": "Orinoquia",
    "97": "Orinoquia",
    "99": "Orinoquia",
    "88": "San_Andres",
}


# =============================================================================
# 0b. HELPER FUNCTIONS
# =============================================================================


def check(msg, df=None, extra=None):
    print("\n" + "=" * 70)
    print(f"  CHECK: {msg}")

    if df is not None:
        print(
            f"  Rows: {len(df):,} | "
            f"Cols: {df.shape[1]} | "
            f"Nulls: {df.isnull().sum().sum():,}"
        )

    if extra:
        print(f"  {extra}")

    print("=" * 70)


def fix_sede_codigo(series):
    """
    Standardize SEDE_CODIGO to a 12-digit string.

    Handles:
    - numeric values
    - scientific notation
    - .0 suffix
    - missing values
    """
    return (
        pd.to_numeric(series, errors="coerce")
        .astype("Int64")
        .astype(str)
        .str.replace("<NA>", "", regex=False)
        .str.replace(r"\.0$", "", regex=True)
        .str.zfill(12)
    )


def normalize_str(s):
    """
    Normalize strings:
    - uppercase
    - remove accents
    - trim spaces
    - collapse multiple spaces
    """
    return (
        s.str.upper()
        .str.strip()
        .apply(
            lambda x: unicodedata.normalize("NFKD", str(x))
            .encode("ascii", errors="ignore")
            .decode("ascii")
        )
        .str.replace(r"\s+", " ", regex=True)
    )


def find_file_flexible(folder, keyword):
    """
    Find first file whose name contains keyword.
    """
    folder = Path(folder)

    if not folder.exists():
        return None

    matches = [
        f for f in folder.glob("*") if keyword.lower() in f.name.lower() and f.is_file()
    ]

    return matches[0] if matches else None


def read_data_safe(path):
    """
    Robust reader for C-600 CSV/Excel files.

    Important:
    Reports malformed lines instead of silently hiding them.
    """

    path = Path(path)

    if not path.exists():
        print(f"  [WARN] File not found: {path.name}")
        return None

    # ---------------------------------------------------------------------
    # Excel
    # ---------------------------------------------------------------------

    if path.suffix.lower() in [".xlsx", ".xls"]:

        try:
            df = pd.read_excel(path)

            df.columns = [
                str(c).strip().replace("\ufeff", "").replace("\u200b", "").upper()
                for c in df.columns
            ]

            return df

        except Exception as e:

            print(f"  [ERROR] Excel read failed {path.name}: {e}")

            return None

    # ---------------------------------------------------------------------
    # CSV
    # ---------------------------------------------------------------------

    for enc in ["utf-8-sig", "utf-8", "cp1252", "latin1"]:

        for sep in [",", ";"]:

            try:

                df = pd.read_csv(
                    path, encoding=enc, sep=sep, low_memory=False, on_bad_lines="warn"
                )

                if len(df.columns) <= 1:
                    continue

                df.columns = [
                    str(c)
                    .strip()
                    .replace("\ufeff", "")
                    .replace("\u200b", "")
                    .strip('"')
                    .strip("'")
                    .upper()
                    for c in df.columns
                ]

                print(
                    f"  [OK] {path.name} | "
                    f"enc:{enc} | "
                    f"sep:'{sep}' | "
                    f"{len(df):,} rows"
                )

                return df

            except UnicodeDecodeError:
                break

            except Exception:
                continue

    print(f"  [ERROR] Could not read: {path.name}")

    return None


# =============================================================================
# 1. LOAD & CONSOLIDATE C-600 — SKELETON
# =============================================================================

print("\n>>> MODULE 1: LOADING C-600 — SKELETON (Jornadas_nivel)")

skeleton_frames = []


for yr in YEARS:

    year_dir = RAW / "C-600" / str(yr)

    if not year_dir.exists():
        year_dir = RAW / "C-600"

    # ---------------------------------------------------------------------
    # Search Jornada / Nivel
    # ---------------------------------------------------------------------

    file_path = find_file_flexible(year_dir, "jornada")

    if file_path is None:

        file_path = find_file_flexible(year_dir, "nivel")

    if file_path is None:

        print(f"  [WARN] No skeleton file found for {yr}")

        print("         Files detected:", [f.name for f in year_dir.glob("*") if f.is_file()])

        continue

    print(f"  Cargando {yr}: {file_path.name}")

    df = read_data_safe(file_path)

    if df is None or df.empty:
        continue

    # ---------------------------------------------------------------------
    # Identify SEDE_CODIGO
    # ---------------------------------------------------------------------

    col_sede = next((c for c in df.columns if "SEDE" in c and "COD" in c), None)

    if not col_sede:

        print(f"  [WARN] No SEDE_CODIGO found in " f"{file_path.name}")

        print("  Columns:", list(df.columns))

        continue

    # ---------------------------------------------------------------------
    # Standardize SEDE
    # ---------------------------------------------------------------------

    df["SEDE_CODIGO"] = fix_sede_codigo(df[col_sede])

    print(f"  [DEBUG] Sedes válidas: " f"{df['SEDE_CODIGO'].notna().sum():,}")

    print(f"  [DEBUG] Sedes únicas: " f"{df['SEDE_CODIGO'].nunique():,}")

    df["PERIODO_ANIO"] = yr

    # ---------------------------------------------------------------------
    # DANE MUNICIPALITY / DEPARTMENT CODES
    #
    # IMPORTANT:
    # We keep the user's current version:
    #
    # municipality = positions 1:6
    # department   = positions 1:3
    #
    # BUT we will validate this later against DIVIPOLA.
    # ---------------------------------------------------------------------

    df["COD_MPIO_DANE"] = df["SEDE_CODIGO"].astype("string").str[1:6]
    df["COD_DPTO_DANE"] = df["SEDE_CODIGO"].astype("string").str[1:3]

    # ---------------------------------------------------------------------
    # MATRICULA TOTAL
    # ---------------------------------------------------------------------

    col_cant = next((c for c in df.columns if "SEDEALUM" in c or "CANTIDAD_TOTAL" in c), None)

    cols_hm = [
        c
        for c in df.columns
        if ("CANTIDAD_HOMBRE" in c or "CANTIDAD_MUJER" in c or "HOMBRE" in c or "MUJER" in c)
    ]

    if col_cant:

        df["MATRICULA_TOTAL"] = pd.to_numeric(df[col_cant], errors="coerce").fillna(0)

    elif cols_hm:

        for c in cols_hm:

            df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)

        df["MATRICULA_TOTAL"] = df[cols_hm].sum(axis=1)

    else:

        print(f"  [WARN] No matrícula column found " f"in {file_path.name}")

        df["MATRICULA_TOTAL"] = 0

    # ---------------------------------------------------------------------
    # Aggregate to SEDE x YEAR
    # ---------------------------------------------------------------------

    agg = (
        df.groupby(
            [
                "SEDE_CODIGO",
                "PERIODO_ANIO",
                "COD_MPIO_DANE",
                "COD_DPTO_DANE",
            ]
        )["MATRICULA_TOTAL"]
        .sum()
        .reset_index()
    )

    skeleton_frames.append(agg)


if not skeleton_frames:

    raise FileNotFoundError("ERROR CRÍTICO: No se logró consolidar ninguna sede.")


panel = pd.concat(skeleton_frames, ignore_index=True)


# -------------------------------------------------------------------------
# Check primary key
# -------------------------------------------------------------------------

duplicate_key = panel.duplicated(subset=["SEDE_CODIGO", "PERIODO_ANIO"], keep=False)


print(f"\n  Duplicate SEDE x YEAR keys: " f"{duplicate_key.sum():,}")


if duplicate_key.any():

    print(panel.loc[duplicate_key, ["SEDE_CODIGO", "PERIODO_ANIO"]].head(20))

    raise ValueError("La llave SEDE_CODIGO x PERIODO_ANIO no es única.")


check(
    "Skeleton built",
    panel,
    (
        f"Unique sedes: "
        f"{panel['SEDE_CODIGO'].nunique():,} | "
        f"Years: "
        f"{sorted(panel['PERIODO_ANIO'].unique())}"
    ),
)


# =============================================================================
# 2. LOAD & JOIN REMAINING C-600 SOURCES
# =============================================================================

print("\n>>> MODULE 2: LOADING & JOINING REMAINING C-600 SOURCES")


C600_CONTEO_COLS = {
    "desplazados": ("JORNDES_CANTIDAD_HOMBRE", "JORNDES_CANTIDAD_MUJER"),
    "limitacion": ("JORNLIM_CANTIDAD_HOMBRE", "JORNLIM_CANTIDAD_MUJER"),
    "tradicional": ("JORNTRA_CANTIDAD_HOMBRE", "JORNTRA_CANTIDAD_MUJER"),
    "flexible": ("JORNMOD_CANTIDAD_HOMBRE", "JORNMOD_CANTIDAD_MUJER"),
    "etnia": ("JORNETN_CANTIDAD_HOMBRE", "JORNETN_CANTIDAD_MUJER"),
}


for key, (col_h, col_m) in C600_CONTEO_COLS.items():

    frames = []

    for yr in YEARS:

        fname = C600_FILES[key]

        path = RAW / "C-600" / str(yr) / f"{fname}_{yr}.csv"

        df = read_data_safe(path)

        if df is None:
            continue

        if "SEDE_CODIGO" not in df.columns:

            print(f"  [WARN] {key} {yr}: " "SEDE_CODIGO not found")

            continue

        df["SEDE_CODIGO"] = fix_sede_codigo(df["SEDE_CODIGO"])

        df["PERIODO_ANIO"] = yr

        # -------------------------------------------------------------
        # Numeric conversion
        # -------------------------------------------------------------

        for c in [col_h, col_m]:

            if c in df.columns:

                df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)

        cols_present = [c for c in [col_h, col_m] if c in df.columns]

        if not cols_present:

            print(f"  [WARN] {key} {yr}: " "count columns not found")

            continue

        # -------------------------------------------------------------
        # Aggregate
        # -------------------------------------------------------------

        agg = (
            df.groupby(["SEDE_CODIGO", "PERIODO_ANIO"])[cols_present]
            .sum()
            .reset_index()
            .rename(
                columns={
                    col_h: f"{key.upper()}_H",
                    col_m: f"{key.upper()}_M",
                }
            )
        )

        h_col = f"{key.upper()}_H"
        m_col = f"{key.upper()}_M"

        if h_col not in agg.columns:
            agg[h_col] = 0

        if m_col not in agg.columns:
            agg[m_col] = 0

        agg[f"{key.upper()}_TOTAL"] = agg[h_col] + agg[m_col]

        frames.append(agg)

    if not frames:

        print(f"  [WARN] {key}: " "no data loaded for any year")

        continue

    df_source = pd.concat(frames, ignore_index=True)

    panel = panel.merge(df_source, on=["SEDE_CODIGO", "PERIODO_ANIO"], how="left")

    match_pct = panel[f"{key.upper()}_TOTAL"].notna().mean() * 100

    print(f"  OK {key:<14}: " f"{match_pct:.1f}% coverage")


check("C-600 lateral joins complete", panel, f"Columns: {panel.shape[1]}")


# =============================================================================
# 3. COMPOSITION & VULNERABILITY
# =============================================================================

print("\n>>> MODULE 3: COMPUTING COMPOSITION & VULNERABILITY")


for key in [
    "DESPLAZADOS",
    "LIMITACION",
    "TRADICIONAL",
    "FLEXIBLE",
    "ETNIA",
]:

    col_total = f"{key}_TOTAL"

    if col_total in panel.columns:

        panel[f"PROP_{key}"] = np.where(
            panel["MATRICULA_TOTAL"] > 0,
            (panel[col_total] / panel["MATRICULA_TOTAL"]).clip(0, 1),
            np.nan,
        )


# -------------------------------------------------------------------------
# Gender index
# -------------------------------------------------------------------------

gender_components = []

for c in [
    "DESPLAZADOS_M",
    "ETNIA_M",
    "TRADICIONAL_M",
    "FLEXIBLE_M",
]:

    if c in panel.columns:
        gender_components.append(panel[c])


if gender_components:

    female_total = sum(gender_components)

    panel["IDX_FEMINIDAD"] = np.where(
        panel["MATRICULA_TOTAL"] > 0, female_total / panel["MATRICULA_TOTAL"], np.nan
    )

else:

    panel["IDX_FEMINIDAD"] = np.nan


# -------------------------------------------------------------------------
# Vulnerability index
# -------------------------------------------------------------------------

weights = {
    "PROP_DESPLAZADOS": 2.0,
    "PROP_LIMITACION": 1.5,
    "PROP_ETNIA": 1.0,
    "PROP_FLEXIBLE": 0.5,
}


panel["INDICE_VULNERABILIDAD"] = 0.0


for c, w in weights.items():

    if c in panel.columns:

        panel["INDICE_VULNERABILIDAD"] += panel[c].fillna(0) * w


# -------------------------------------------------------------------------
# Pandemic
# -------------------------------------------------------------------------

panel["FLAG_PANDEMIA"] = (panel["PERIODO_ANIO"] == 2020).astype(int)


check(
    "Composition features computed",
    panel,
    (f"Mean vulnerability: " f"{panel['INDICE_VULNERABILIDAD'].mean():.4f}"),
)


# =============================================================================
# 4. SIMAT
# =============================================================================

print("\n>>> MODULE 4: LOADING SIMAT")


def load_simat(filename, conteo_col):

    path = RAW / "SIMAT" / filename

    if not path.exists():

        print(f"  [ERROR] Not found: {filename}")

        return pd.DataFrame()

    df = pd.read_excel(path, sheet_name=0, skiprows=5)

    df.columns = df.columns.str.strip().str.upper().str.replace(" ", "_")

    df = df.rename(columns={"AÑO": "ANIO", "A_O": "ANIO"})

    # Find year column
    yr_col = next((c for c in df.columns if "A" in c and "O" in c and len(c) <= 4), None)

    if yr_col and yr_col != "ANIO":

        df = df.rename(columns={yr_col: "ANIO"})

    df["ANIO"] = pd.to_numeric(df.get("ANIO", np.nan), errors="coerce")

    # -------------------------------------------------------------
    # Restrict strictly to panel years
    # -------------------------------------------------------------

    df = df[
        (df.get("TERRITORIO", "") == "MUNICIPIO")
        & (df.get("SECTOR", "").str.lower().str.contains("oficial", na=False))
        & (df["ANIO"].isin(YEARS))
    ].copy()

    df["TASA"] = pd.to_numeric(df.get("TASA", np.nan), errors="coerce")

    df["TOTAL"] = pd.to_numeric(df.get("TOTAL", np.nan), errors="coerce")

    df[conteo_col] = pd.to_numeric(df.get(conteo_col, np.nan), errors="coerce")

    df["MUNICIPIO_STD"] = normalize_str(df["MUNICIPIO"].fillna(""))

    df["DEPARTAMENTO_STD"] = normalize_str(df["DEPARTAMENTO"].fillna(""))

    return df[
        [
            "ANIO",
            "MUNICIPIO_STD",
            "DEPARTAMENTO_STD",
            conteo_col,
            "TOTAL",
            "TASA",
        ]
    ]


df_desercion = load_simat("Tasa_Desercion_intra_Departamentos.xlsx", "DESERTORES")


df_repitencia = load_simat("Tasa_repitencia_intra_Departamentos.xlsx", "REPITENTES")


check(
    "SIMAT loaded",
    df_desercion,
    (
        f"Desercion rows: "
        f"{len(df_desercion):,} | "
        f"Repitencia rows: "
        f"{len(df_repitencia):,}"
    ),
)


# =============================================================================
# 4b. DIVIPOLA
# =============================================================================

print("  Loading DIVIPOLA...")


divipola_path = RAW / "SIMAT" / "DIVIPOLA.csv"


div = pd.read_csv(divipola_path, encoding="utf-8-sig", low_memory=False)


div.columns = [
    unicodedata.normalize("NFKD", str(c))
    .encode("ascii", errors="ignore")
    .decode("utf-8")
    .upper()
    .strip()
    for c in div.columns
]


cod_col = next((c for c in div.columns if "COD" in c and ("MUN" in c or "MPO" in c)), None)


nom_col = next((c for c in div.columns if ("MUN" in c or "MPO" in c) and "COD" not in c), None)


dpto_col = next(
    (c for c in div.columns if ("DEP" in c or "DPTO" in c) and "COD" not in c), None
)


if not all([cod_col, nom_col, dpto_col]):

    raise ValueError("No se pudieron identificar " "las columnas de DIVIPOLA.")


div["COD_MPIO_DANE"] = div[cod_col].astype(str).str.zfill(5)


div["MUNICIPIO_STD"] = normalize_str(div[nom_col].fillna(""))


div["DEPARTAMENTO_STD"] = normalize_str(div[dpto_col].fillna(""))


divipola_clean = div[["COD_MPIO_DANE", "MUNICIPIO_STD", "DEPARTAMENTO_STD"]].drop_duplicates()


check(
    "DIVIPOLA loaded",
    divipola_clean,
    (f"Unique municipalities: " f"{divipola_clean['COD_MPIO_DANE'].nunique():,}"),
)


# =============================================================================
# 4c. AGGREGATE SIMAT
# =============================================================================


def agg_simat_to_mpio(df_simat, tasa_col_out, conteo_col):

    def wavg(g):

        mask = g["TASA"].notna() & g["TOTAL"].notna() & (g["TOTAL"] > 0)

        if mask.sum() == 0:

            return pd.Series(
                {tasa_col_out: np.nan, f"{conteo_col}_SUM": np.nan, "TOTAL_SUM": np.nan}
            )

        return pd.Series(
            {
                tasa_col_out: np.average(g.loc[mask, "TASA"], weights=g.loc[mask, "TOTAL"]),
                f"{conteo_col}_SUM": g[conteo_col].sum(),
                "TOTAL_SUM": g["TOTAL"].sum(),
            }
        )

    return (
        df_simat.groupby(["ANIO", "MUNICIPIO_STD", "DEPARTAMENTO_STD"])
        .apply(wavg)
        .reset_index()
    )


simat_desercion = agg_simat_to_mpio(df_desercion, "TASA_DESERCION_MPIO", "DESERTORES")


simat_repitencia = agg_simat_to_mpio(df_repitencia, "TASA_REPITENCIA_MPIO", "REPITENTES")


# -------------------------------------------------------------------------
# Match DIVIPOLA
# -------------------------------------------------------------------------

simat_desercion = simat_desercion.merge(
    divipola_clean, on=["MUNICIPIO_STD", "DEPARTAMENTO_STD"], how="left"
)


simat_repitencia = simat_repitencia.merge(
    divipola_clean, on=["MUNICIPIO_STD", "DEPARTAMENTO_STD"], how="left"
)


no_match_d = simat_desercion["COD_MPIO_DANE"].isna().sum()


no_match_r = simat_repitencia["COD_MPIO_DANE"].isna().sum()


check(
    "SIMAT aggregated & DIVIPOLA matched",
    extra=(f"Desercion no-match: {no_match_d} | " f"Repitencia no-match: {no_match_r}"),
)


# -------------------------------------------------------------------------
# Merge SIMAT to panel
# -------------------------------------------------------------------------

panel = panel.merge(
    simat_desercion[["ANIO", "COD_MPIO_DANE", "TASA_DESERCION_MPIO"]],
    left_on=["PERIODO_ANIO", "COD_MPIO_DANE"],
    right_on=["ANIO", "COD_MPIO_DANE"],
    how="left",
).drop(columns=["ANIO"], errors="ignore")


panel = panel.merge(
    simat_repitencia[["ANIO", "COD_MPIO_DANE", "TASA_REPITENCIA_MPIO"]],
    left_on=["PERIODO_ANIO", "COD_MPIO_DANE"],
    right_on=["ANIO", "COD_MPIO_DANE"],
    how="left",
).drop(columns=["ANIO"], errors="ignore")


cov_d = panel["TASA_DESERCION_MPIO"].notna().mean() * 100


cov_r = panel["TASA_REPITENCIA_MPIO"].notna().mean() * 100


check(
    "SIMAT imputed to panel",
    extra=(f"Desercion coverage: {cov_d:.1f}% | " f"Repitencia coverage: {cov_r:.1f}%"),
)


# =============================================================================
# 5. TEMPORAL FEATURES
# =============================================================================

print("\n>>> MODULE 5: COMPUTING TEMPORAL FEATURES & LAGS")


panel = panel.sort_values(["SEDE_CODIGO", "PERIODO_ANIO"]).reset_index(drop=True)


# -------------------------------------------------------------------------
# Train / Test split
# -------------------------------------------------------------------------

panel["SAMPLE"] = np.where(panel["PERIODO_ANIO"] == TEST_YEAR, "TEST", "TRAIN")


# -------------------------------------------------------------------------
# Enrollment dynamics
# -------------------------------------------------------------------------

panel["MATRICULA_DELTA"] = panel.groupby("SEDE_CODIGO")["MATRICULA_TOTAL"].diff()


panel["MATRICULA_PCT_CAMBIO"] = (
    panel["MATRICULA_DELTA"] / panel.groupby("SEDE_CODIGO")["MATRICULA_TOTAL"].shift(1)
).clip(-1, 5)


# -------------------------------------------------------------------------
# Lagged dropout / repetition
# -------------------------------------------------------------------------

panel["DESERCION_LAG1"] = panel.groupby("SEDE_CODIGO")["TASA_DESERCION_MPIO"].shift(1)


panel["REPITENCIA_LAG1"] = panel.groupby("SEDE_CODIGO")["TASA_REPITENCIA_MPIO"].shift(1)


# -------------------------------------------------------------------------
# IMPORTANT:
# MA2 LAGGED
#
# Previous version:
# rolling(2) included current-year dropout.
#
# New version:
# shift(1) FIRST, then rolling(2).
#
# Therefore, for 2023:
# DESERCION_MA2_LAG uses 2021 + 2022.
# It does NOT use 2023.
# -------------------------------------------------------------------------

panel["DESERCION_MA2_LAG"] = panel.groupby("SEDE_CODIGO")["TASA_DESERCION_MPIO"].transform(
    lambda x: x.shift(1).rolling(2, min_periods=1).mean()
)


# -------------------------------------------------------------------------
# Two consecutive years of declining enrollment
# -------------------------------------------------------------------------

panel["FLAG_DECLIVE_MATRICULA"] = (
    (panel["MATRICULA_DELTA"] < 0)
    & (panel.groupby("SEDE_CODIGO")["MATRICULA_DELTA"].shift(1) < 0)
).astype(int)


check(
    "Temporal features computed",
    panel,
    (
        f"Lag1 dropout coverage: "
        f"{panel['DESERCION_LAG1'].notna().sum():,} | "
        f"2023 test rows: "
        f"{(panel['SAMPLE'] == 'TEST').sum():,}"
    ),
)


# =============================================================================
# 6. IPM — REGIONAL CONTEXT
# =============================================================================

print("\n>>> MODULE 6: LOADING IPM")


ipm_frames = []


for yr in YEARS:

    path = RAW / "IPM" / f"IPM_Hogares_{yr}.csv"

    if not path.exists():

        print(f"  [WARN] IPM {yr} not found")

        continue

    for sep_try in [";", ","]:
        df = pd.read_csv(path, sep=sep_try, encoding="latin1", low_memory=False)
        if len(df.columns) > 5:
            break
    df.columns = df.columns.str.strip().str.lower()
    df["anio"] = yr
    ipm_frames.append(df)

    df.columns = df.columns.str.strip().str.lower()

    df["anio"] = yr

    ipm_frames.append(df)


if ipm_frames:

    df_ipm = pd.concat(ipm_frames, ignore_index=True)

else:

    df_ipm = pd.DataFrame()


if not df_ipm.empty:

    df_ipm["region"] = pd.to_numeric(df_ipm.get("region", np.nan), errors="coerce")

    df_ipm["region_nombre"] = df_ipm["region"].map(REGION_MAP)

    privaciones = [
        "inasistencia_escolar",
        "rezago_escolar",
        "trabajo_infantil",
        "hacinamiento",
        "empleo_formal",
        "alfabetismo",
        "logro_educativo",
        "aseguramiento_salud",
        "barreras_acceso_salud",
        "paredes",
        "pisos",
        "alcantarillado",
        "acueducto",
        "desempleo_larga_duracion",
        "atencion_integral",
        "ipm",
    ]

    cols_ipm = [c for c in privaciones if c in df_ipm.columns]

    df_ipm["fex_c"] = pd.to_numeric(df_ipm.get("fex_c", 1), errors="coerce").fillna(1)

    def weighted_ipm(g):

        result = {}

        for c in cols_ipm:

            values = pd.to_numeric(g[c], errors="coerce").fillna(0)

            result[f"IPM_{c.upper()}"] = np.average(values, weights=g["fex_c"])

        return pd.Series(result)

    ipm_agg = (
        df_ipm.groupby(["anio", "region_nombre"])
        .apply(weighted_ipm)
        .reset_index()
        .rename(columns={"anio": "PERIODO_ANIO", "region_nombre": "REGION_DANE"})
    )

    panel["REGION_DANE"] = panel["COD_DPTO_DANE"].map(DPTO_TO_REGION)

    panel = panel.merge(ipm_agg, on=["PERIODO_ANIO", "REGION_DANE"], how="left")

    ipm_coverage = panel["IPM_IPM"].notna().mean() * 100

    check("IPM merged", panel, f"IPM coverage: {ipm_coverage:.1f}%")


else:

    print("  [WARN] No IPM data available.")


# =============================================================================
# 7. ENRICHMENT — PDET & ZOMAC
# =============================================================================
print("\n>>> MODULE 7: LOADING PDET & ZOMAC")

# -----------------------------------------------------------------------------
# PDET
# -----------------------------------------------------------------------------
pdet_path = RAW / "Enrichment" / "PDET_municipios.xlsx"

if pdet_path.exists():

    pdet = pd.read_excel(pdet_path)
    pdet.columns = pdet.columns.astype(str).str.strip().str.upper()

    # The municipality identifier is explicitly COD DANE.
    if "COD DANE" not in pdet.columns:
        raise KeyError(
            f"PDET: expected column 'COD DANE'. " f"Available columns: {pdet.columns.tolist()}"
        )

    pdet["COD_MPIO_DANE"] = (
        pd.to_numeric(pdet["COD DANE"], errors="coerce")
        .astype("Int64")
        .astype(str)
        .str.replace("<NA>", "", regex=False)
        .str.zfill(5)
    )

    # Remove invalid codes
    pdet = pdet[pdet["COD_MPIO_DANE"].str.fullmatch(r"\d{5}", na=False)].copy()

    # One observation per municipality
    pdet_codes = pdet[["COD_MPIO_DANE"]].drop_duplicates().assign(FLAG_PDET=1)

    panel = panel.merge(pdet_codes, on="COD_MPIO_DANE", how="left", validate="many_to_one")

    panel["FLAG_PDET"] = panel["FLAG_PDET"].fillna(0).astype(int)

    print(
        f"  OK PDET: "
        f"{pdet['COD_MPIO_DANE'].nunique():,} municipalities | "
        f"{panel['FLAG_PDET'].sum():,} sede-years matched"
    )

else:
    print(f"  [WARN] PDET file not found: {pdet_path}")


# -----------------------------------------------------------------------------
# ZOMAC
# -----------------------------------------------------------------------------
zomac_path = RAW / "Enrichment" / "ZOMAC_municipios.xlsx"

if zomac_path.exists():

    zomac = pd.read_excel(zomac_path)
    zomac.columns = zomac.columns.astype(str).str.strip().str.upper()

    # The municipality identifier is explicitly COD DANE.
    if "COD DANE" not in zomac.columns:
        raise KeyError(
            f"ZOMAC: expected column 'COD DANE'. "
            f"Available columns: {zomac.columns.tolist()}"
        )

    zomac["COD_MPIO_DANE"] = (
        pd.to_numeric(zomac["COD DANE"], errors="coerce")
        .astype("Int64")
        .astype(str)
        .str.replace("<NA>", "", regex=False)
        .str.zfill(5)
    )

    # Remove invalid codes
    zomac = zomac[zomac["COD_MPIO_DANE"].str.fullmatch(r"\d{5}", na=False)].copy()

    # One observation per municipality
    zomac_codes = zomac[["COD_MPIO_DANE"]].drop_duplicates().assign(FLAG_ZOMAC=1)

    panel = panel.merge(zomac_codes, on="COD_MPIO_DANE", how="left", validate="many_to_one")

    panel["FLAG_ZOMAC"] = panel["FLAG_ZOMAC"].fillna(0).astype(int)

    print(
        f"  OK ZOMAC: "
        f"{zomac['COD_MPIO_DANE'].nunique():,} municipalities | "
        f"{panel['FLAG_ZOMAC'].sum():,} sede-years matched"
    )

else:
    print(f"  [WARN] ZOMAC file not found: {zomac_path}")


# -----------------------------------------------------------------------------
# Basic validation
# -----------------------------------------------------------------------------
print("\n  PDET / ZOMAC validation:")

if "FLAG_PDET" in panel.columns:
    print(f"    PDET-positive sede-years : " f"{panel['FLAG_PDET'].sum():,}")

if "FLAG_ZOMAC" in panel.columns:
    print(f"    ZOMAC-positive sede-years: " f"{panel['FLAG_ZOMAC'].sum():,}")

if "FLAG_PDET" in panel.columns and "FLAG_ZOMAC" in panel.columns:
    print(
        f"    Both PDET & ZOMAC       : "
        f"{((panel['FLAG_PDET'] == 1) & (panel['FLAG_ZOMAC'] == 1)).sum():,}"
    )

# =============================================================================
# PDET / ZOMAC KEY DIAGNOSTIC
# =============================================================================
print("\n" + "=" * 80)
print("DIAGNÓSTICO PDET / ZOMAC")
print("=" * 80)

# --- PDET ---
print("\n### PDET ###")
print(f"Municipios en archivo : {pdet['COD_MPIO_DANE'].nunique():,}")
print(
    f"Códigos válidos       : "
    f"{pdet['COD_MPIO_DANE'].str.fullmatch(r'\\d{{5}}', na=False).sum():,}"
)

print("\nPrimeros códigos PDET:")
print(pdet[["COD DANE", "COD_MPIO_DANE", "MUNICIPIO"]].head(10).to_string(index=False))

# --- ZOMAC ---
print("\n### ZOMAC ###")
print(f"Municipios en archivo : {zomac['COD_MPIO_DANE'].nunique():,}")
print(
    f"Códigos válidos       : "
    f"{zomac['COD_MPIO_DANE'].str.fullmatch(r'\\d{{5}}', na=False).sum():,}"
)

print("\nPrimeros códigos ZOMAC:")
print(zomac[["COD DANE", "COD_MPIO_DANE", "MUNICIPIO"]].head(10).to_string(index=False))

# --- Panel ---
print("\n### PANEL ###")
panel_mpios = panel[["COD_MPIO_DANE"]].dropna().astype(str).drop_duplicates()

print(f"Municipios únicos en panel: " f"{panel_mpios['COD_MPIO_DANE'].nunique():,}")

pdet_panel_matches = panel_mpios["COD_MPIO_DANE"].isin(set(pdet["COD_MPIO_DANE"])).sum()

zomac_panel_matches = panel_mpios["COD_MPIO_DANE"].isin(set(zomac["COD_MPIO_DANE"])).sum()

print(f"Municipios panel que aparecen en PDET : " f"{pdet_panel_matches:,}")

print(f"Municipios panel que aparecen en ZOMAC: " f"{zomac_panel_matches:,}")

print("\n### FLAGS EN PANEL ###")

print(panel["FLAG_PDET"].value_counts(dropna=False).sort_index().rename("N").to_string())

print()

print(panel["FLAG_ZOMAC"].value_counts(dropna=False).sort_index().rename("N").to_string())

print("\n" + "=" * 80)
print("FIN DIAGNÓSTICO PDET / ZOMAC")
print("=" * 80)

# =============================================================================
# 7c. ICFES
# =============================================================================


def normalizar_codigo_12(series: pd.Series) -> pd.Series:
    """Normaliza identificadores DANE a cadenas de texto de exactamente 12 dígitos."""

    def _limpiar(val):
        if pd.isna(val):
            return pd.NA
        s = str(val).strip()
        if s.lower() in ["nan", "na", "n/a", "null", "none", ""]:
            return pd.NA
        if "e" in s.lower():
            try:
                s = f"{float(s):.0f}"
            except ValueError:
                return pd.NA
        s = re.sub(r"\.0+$", "", s)
        s = "".join(filter(str.isdigit, s))
        if not s:
            return pd.NA
        s = s.zfill(12)
        return s if len(s) == 12 else pd.NA

    return series.apply(_limpiar)


icfes_path = RAW / "Enrichment" / "Icfes_Resumen.csv"
if icfes_path.exists():
    # 1. Cargar datos con soporte UTF-8 (BOM generado por R)
    try:
        icfes = pd.read_csv(icfes_path, encoding="utf-8-sig", low_memory=False)
    except Exception:
        icfes = pd.read_csv(icfes_path, encoding="latin1", low_memory=False)

    icfes.columns = icfes.columns.str.strip().str.upper()

    # 2. Normalización de identificadores
    icfes["COD_SEDE_ICFES"] = normalizar_codigo_12(icfes.get("COLE_COD_DANE_SEDE"))
    icfes["COD_ESTABLECIMIENTO"] = normalizar_codigo_12(
        icfes.get("COLE_COD_DANE_ESTABLECIMIENTO")
    )

    if "ANIO" in icfes.columns:
        icfes["PERIODO_ANIO"] = pd.to_numeric(icfes["ANIO"], errors="coerce")
    elif "PERIODO" in icfes.columns:
        icfes["PERIODO_ANIO"] = pd.to_numeric(
            icfes["PERIODO"].astype(str).str[:4], errors="coerce"
        )

    # Normalizar llave en Panel de Observaciones
    panel_raw = panel["SEDE_CODIGO"].astype(str).str.strip()
    panel["_SEDE_KEY"] = panel_raw.apply(
        lambda x: x[1:] if len(x) == 13 and x.startswith("1") else x
    )
    panel["_SEDE_KEY"] = normalizar_codigo_12(panel["_SEDE_KEY"])

    # 3. Mapeo completo de variables nuevas desde Icfes_Resumen.csv
    icfes_cols = {
        "CANT_ESTUDIANTES": "ICFES_CANT_ESTUDIANTES",
        "PROM_PUNT_GLOBAL": "ICFES_PROM_PUNT_GLOBAL",
        "PROM_PUNT_LECTURA": "ICFES_PROM_LECTURA",
        "PROM_PUNT_MATEMATICAS": "ICFES_PROM_MATEMATICAS",
        "PROM_PUNT_CIENCIAS": "ICFES_PROM_CIENCIAS",
        "PROM_PUNT_SOCIALES": "ICFES_PROM_SOCIALES",
        "PROM_PUNT_INGLES": "ICFES_PROM_INGLES",
        "PROM_INSE_INDIVIDUAL": "ICFES_PROM_INSE",
        "PCT_ESTRATO_1_2": "ICFES_PCT_ESTRATO_1_2",
        "PCT_FAMI_TIENEINTERNET": "ICFES_PCT_INTERNET",
        "PCT_FAMI_TIENECOMPUTADOR": "ICFES_PCT_COMPUTADOR",
        "PCT_DESPLAZACOLEGIO": "ICFES_PCT_DESPLAZACOLEGIO",
        "PCT_HORASTRABNOREMU": "ICFES_PCT_HORASTRABNOREMU",
        "PCT_COMUNIDADCAMPESINA": "ICFES_PCT_CAMPESINA",
    }

    cols_present = {k: v for k, v in icfes_cols.items() if k in icfes.columns}
    target_cols = list(cols_present.values())

    # 4. Tabla 1: Nivel SEDE + AÑO
    icfes_sede = (
        icfes.dropna(subset=["COD_SEDE_ICFES", "PERIODO_ANIO"])[
            ["COD_SEDE_ICFES", "PERIODO_ANIO"] + list(cols_present.keys())
        ]
        .rename(columns=cols_present)
        .drop_duplicates(subset=["COD_SEDE_ICFES", "PERIODO_ANIO"])
    )

    # 5. Tabla 2: Nivel ESTABLECIMIENTO + AÑO (Agregado ponderado por estudiantes)
    icfes_est_raw = icfes.dropna(subset=["COD_ESTABLECIMIENTO", "PERIODO_ANIO"]).copy()

    # Definir reglas de agregación por establecimiento
    agg_rules = {}
    for orig_col in cols_present.keys():
        if orig_col == "CANT_ESTUDIANTES":
            agg_rules[orig_col] = "sum"
        else:
            agg_rules[orig_col] = "mean"  # Promedio entre sedes

    icfes_est = (
        icfes_est_raw.groupby(["COD_ESTABLECIMIENTO", "PERIODO_ANIO"])[
            list(cols_present.keys())
        ]
        .agg(agg_rules)
        .reset_index()
        .rename(columns=cols_present)
    )

    # 6. PASO 1: Merge por SEDE
    n_total = len(panel)
    panel = panel.merge(
        icfes_sede,
        left_on=["_SEDE_KEY", "PERIODO_ANIO"],
        right_on=["COD_SEDE_ICFES", "PERIODO_ANIO"],
        how="left",
    ).drop(columns=["COD_SEDE_ICFES"])

    n_match_sede = panel["ICFES_PROM_PUNT_GLOBAL"].notna().sum()

    # 7. PASO 2: Rescate por ESTABLECIMIENTO
    unmatched_mask = panel["ICFES_PROM_PUNT_GLOBAL"].isna()
    n_unmatched_pre = unmatched_mask.sum()
    n_rescued = 0

    if n_unmatched_pre > 0:
        cols_to_keep = [c for c in panel.columns if c not in target_cols]
        rescued = panel.loc[unmatched_mask, cols_to_keep].merge(
            icfes_est,
            left_on=["_SEDE_KEY", "PERIODO_ANIO"],
            right_on=["COD_ESTABLECIMIENTO", "PERIODO_ANIO"],
            how="left",
        )

        n_rescued = rescued["ICFES_PROM_PUNT_GLOBAL"].notna().sum()

        # Asignar los valores del rescate al panel principal
        for col in target_cols:
            panel.loc[unmatched_mask, col] = rescued[col].values

    panel = panel.drop(columns=["_SEDE_KEY"])

    # 8. Reporte consolidado
    n_final_match = panel["ICFES_PROM_PUNT_GLOBAL"].notna().sum()
    pct_cobertura = (n_final_match / n_total) * 100 if n_total > 0 else 0

    print("\n" + "=" * 55)
    print("REPORTE DE INTEGRACIÓN ICFES AL PANEL")
    print("=" * 55)
    print(f" Total observaciones en Panel : {n_total:,}")
    print(f" Matches directos por SEDE : {n_match_sede:,} ({n_match_sede/n_total*100:.2f}%)")
    print(f" Rescatados por ESTABLECIM. : {n_rescued:,} ({n_rescued/n_total*100:.2f}%)")
    print(f"  Registros sin ICFES       : {n_total - n_final_match:,}")
    print(f"  COBERTURA FINAL           : {pct_cobertura:.2f}%")
    print("=" * 55)

# =============================================================================
# 8. KEY / GEOGRAPHIC DIAGNOSTIC
#
# This is the most important new diagnostic.
#
# We compare:
#
# Current:
#   municipality = SEDE[1:6]
#   department   = SEDE[1:3]
#
# This lets us decide empirically which interpretation is correct.
# =============================================================================

print("\n>>> MODULE 8: DIAGNOSTIC OF DANE KEYS")


# -------------------------------------------------------------------------
# Candidate codes
# -------------------------------------------------------------------------

panel["COD_MPIO_CAND_CURRENT"] = panel["SEDE_CODIGO"].astype("string").str[1:6]


panel["COD_DPTO_CAND_CURRENT"] = panel["SEDE_CODIGO"].astype("string").str[1:3]


panel["COD_MPIO_CAND_STANDARD"] = panel["SEDE_CODIGO"].astype("string").str[:5]


panel["COD_DPTO_CAND_STANDARD"] = panel["SEDE_CODIGO"].astype("string").str[:2]


div_codes = set(divipola_clean["COD_MPIO_DANE"].dropna().astype(str))


current_match = panel["COD_MPIO_CAND_CURRENT"].isin(div_codes)


standard_match = panel["COD_MPIO_CAND_STANDARD"].isin(div_codes)


print("\n--- MUNICIPAL CODE VALIDATION ---")


print(f"Current [1:6] valid DIVIPOLA: " f"{current_match.mean()*100:.2f}%")


print(f"Standard [:5] valid DIVIPOLA: " f"{standard_match.mean()*100:.2f}%")


# -------------------------------------------------------------------------
# PDET validation
# -------------------------------------------------------------------------

if "pdet" in locals():

    pdet_codes = set(pdet["COD_MPIO_DANE"].dropna().astype(str))

    pdet_current = panel["COD_MPIO_CAND_CURRENT"].isin(pdet_codes).sum()

    pdet_standard = panel["COD_MPIO_CAND_STANDARD"].isin(pdet_codes).sum()

    print("\n--- PDET VALIDATION ---")

    print(f"PDET matches with [1:6]: " f"{pdet_current:,}")

    print(f"PDET matches with [:5]: " f"{pdet_standard:,}")


# -------------------------------------------------------------------------
# ZOMAC validation
# -------------------------------------------------------------------------

if "zomac" in locals():

    zomac_codes = set(zomac["COD_MPIO_DANE"].dropna().astype(str))

    zomac_current = panel["COD_MPIO_CAND_CURRENT"].isin(zomac_codes).sum()

    zomac_standard = panel["COD_MPIO_CAND_STANDARD"].isin(zomac_codes).sum()

    print("\n--- ZOMAC VALIDATION ---")

    print(f"ZOMAC matches with [1:6]: " f"{zomac_current:,}")

    print(f"ZOMAC matches with [:5]: " f"{zomac_standard:,}")


# -------------------------------------------------------------------------
# Display examples
# -------------------------------------------------------------------------

print("\n--- CODE EXAMPLES ---")


print(
    panel[
        [
            "SEDE_CODIGO",
            "COD_MPIO_CAND_CURRENT",
            "COD_MPIO_CAND_STANDARD",
            "COD_DPTO_CAND_CURRENT",
            "COD_DPTO_CAND_STANDARD",
        ]
    ]
    .drop_duplicates()
    .head(20)
    .to_string(index=False)
)


# -------------------------------------------------------------------------
# Keep current version for now.
#
# We will decide whether to change this after seeing diagnostics.
# -------------------------------------------------------------------------

panel["COD_MPIO_DANE"] = panel["COD_MPIO_CAND_CURRENT"]


panel["COD_DPTO_DANE"] = panel["COD_DPTO_CAND_CURRENT"]


# Drop diagnostic candidate columns
panel = panel.drop(
    columns=[
        "COD_MPIO_CAND_CURRENT",
        "COD_DPTO_CAND_CURRENT",
        "COD_MPIO_CAND_STANDARD",
        "COD_DPTO_CAND_STANDARD",
    ],
    errors="ignore",
)


# =============================================================================
# 9. FINAL CLEANUP
# =============================================================================

print("\n>>> MODULE 9: FINAL CLEANUP")


# -------------------------------------------------------------------------
# Remove technical columns
# -------------------------------------------------------------------------

drop_patterns = [
    "_ID",
    "_CODIGO",
    "PERIODO_ID",
]


cols_to_drop = [
    c for c in panel.columns if any(p in c for p in drop_patterns) and c != "SEDE_CODIGO"
]


panel = panel.drop(columns=cols_to_drop, errors="ignore")


# -------------------------------------------------------------------------
# Logical column groups
# -------------------------------------------------------------------------

cols_key = [
    "SEDE_CODIGO",
    "PERIODO_ANIO",
    "SAMPLE",
    "COD_MPIO_DANE",
    "COD_DPTO_DANE",
    "REGION_DANE",
]


cols_target = [
    "TASA_DESERCION_MPIO",
    "TASA_REPITENCIA_MPIO",
]


cols_lags = [
    "DESERCION_LAG1",
    "REPITENCIA_LAG1",
    "DESERCION_MA2_LAG",
]


cols_matricula = [
    c
    for c in panel.columns
    if ("MATRICULA" in c or c == "FLAG_PANDEMIA" or c == "FLAG_DECLIVE_MATRICULA")
]


cols_c600 = [
    c
    for c in panel.columns
    if any(
        k in c
        for k in [
            "DESPLAZADOS",
            "LIMITACION",
            "TRADICIONAL",
            "FLEXIBLE",
            "ETNIA",
            "PROP_",
            "INDICE",
            "IDX_",
        ]
    )
]


cols_icfes = [c for c in panel.columns if c.startswith("ICFES_")]


cols_ipm = [c for c in panel.columns if c.startswith("IPM_")]


cols_enrichment = [c for c in panel.columns if c in ["FLAG_PDET", "FLAG_ZOMAC"]]


cols_rest = [
    c
    for c in panel.columns
    if c
    not in (
        cols_key
        + cols_target
        + cols_lags
        + cols_matricula
        + cols_c600
        + cols_icfes
        + cols_ipm
        + cols_enrichment
    )
]


# -------------------------------------------------------------------------
# Final ordering
# -------------------------------------------------------------------------

final_order = (
    cols_key
    + cols_target
    + cols_lags
    + cols_matricula
    + cols_c600
    + cols_icfes
    + cols_ipm
    + cols_enrichment
    + cols_rest
)


# Remove duplicated names
final_order = list(dict.fromkeys(final_order))


# Keep existing columns
final_order = [c for c in final_order if c in panel.columns]


panel = panel.loc[:, final_order]


panel = panel.sort_values(["SEDE_CODIGO", "PERIODO_ANIO"]).reset_index(drop=True)


# =============================================================================
# 10. FINAL VALIDATION
# =============================================================================

print("\n>>> MODULE 10: FINAL VALIDATION")


# -------------------------------------------------------------------------
# Duplicate columns
# -------------------------------------------------------------------------

duplicated_cols = panel.columns[panel.columns.duplicated(keep=False)].tolist()


if duplicated_cols:

    print("[ERROR] Duplicate columns:")

    print(duplicated_cols)

    raise ValueError("El panel contiene columnas duplicadas.")


# -------------------------------------------------------------------------
# Duplicate primary key
# -------------------------------------------------------------------------

duplicate_key = panel.duplicated(subset=["SEDE_CODIGO", "PERIODO_ANIO"], keep=False)


if duplicate_key.any():

    print("[ERROR] Duplicate SEDE x YEAR:")

    print(panel.loc[duplicate_key, ["SEDE_CODIGO", "PERIODO_ANIO"]].head(20))

    raise ValueError("La llave SEDE_CODIGO x PERIODO_ANIO no es única.")


# -------------------------------------------------------------------------
# Period validation
# -------------------------------------------------------------------------

unexpected_years = sorted(set(panel["PERIODO_ANIO"].dropna().astype(int)) - set(YEARS))


if unexpected_years:

    raise ValueError(f"Existen años fuera del período permitido: " f"{unexpected_years}")


# -------------------------------------------------------------------------
# Train / test counts
# -------------------------------------------------------------------------

train_rows = (panel["SAMPLE"] == "TRAIN").sum()


test_rows = (panel["SAMPLE"] == "TEST").sum()


print(f"  TRAIN rows: {train_rows:,}")


print(f"  TEST rows:  {test_rows:,}")


print(f"  TRAIN years: {TRAIN_YEARS}")


print(f"  TEST year:   {TEST_YEAR}")


print(f"  Unique sedes: " f"{panel['SEDE_CODIGO'].nunique():,}")


print(f"  Total rows: " f"{len(panel):,}")


print(f"  Total columns: " f"{len(panel.columns):,}")


print("\n[OK] Final validation passed.")


# =============================================================================
# 11. QUALITY DIAGNOSTICS
# =============================================================================

print("\n>>> MODULE 11: GENERATING QUALITY REPORT")


# -------------------------------------------------------------------------
# General completeness
# -------------------------------------------------------------------------

diag = pd.DataFrame(
    {
        "Variable": panel.columns,
        "Tipo": panel.dtypes.astype(str).values,
        "N_nulos": panel.isnull().sum().values,
        "Pct_nulos": (panel.isnull().mean() * 100).round(2).values,
        "Media": [
            panel[c].mean() if pd.api.types.is_numeric_dtype(panel[c]) else None
            for c in panel.columns
        ],
        "Min": [
            panel[c].min() if pd.api.types.is_numeric_dtype(panel[c]) else None
            for c in panel.columns
        ],
        "Max": [
            panel[c].max() if pd.api.types.is_numeric_dtype(panel[c]) else None
            for c in panel.columns
        ],
    }
)


diag = diag.sort_values("Pct_nulos", ascending=False)


# =============================================================================
# 11b. COVERAGE BY YEAR
# =============================================================================

coverage_variables = [
    "MATRICULA_TOTAL",
    "TASA_DESERCION_MPIO",
    "TASA_REPITENCIA_MPIO",
    "DESERCION_LAG1",
    "REPITENCIA_LAG1",
    "DESERCION_MA2_LAG",
    "IPM_IPM",
    "FLAG_PDET",
    "FLAG_ZOMAC",
    "ICFES_PROM_PUNT_GLOBAL",
]


coverage_variables = [c for c in coverage_variables if c in panel.columns]


coverage_by_year = (
    panel.groupby("PERIODO_ANIO")[coverage_variables]
    .apply(lambda x: x.notna().mean() * 100)
    .reset_index()
)


# =============================================================================
# 11c. COVERAGE TRAIN VS TEST
# =============================================================================

coverage_train_test = (
    panel.groupby("SAMPLE")[coverage_variables]
    .apply(lambda x: x.notna().mean() * 100)
    .reset_index()
)


# =============================================================================
# 11d. TARGET CORRELATIONS
# =============================================================================

target = "TASA_DESERCION_MPIO"


num_cols = panel.select_dtypes(include=[np.number]).columns.tolist()


num_cols = [c for c in num_cols if (c != target and panel[c].notna().sum() > 500)]


corr = (
    panel[num_cols + [target]]
    .corr()[target]
    .drop(target, errors="ignore")
    .sort_values(key=abs, ascending=False)
    .reset_index()
    .rename(columns={"index": "Variable", target: "Correlacion_Pearson"})
)


corr["Correlacion_Pearson"] = corr["Correlacion_Pearson"].round(4)


print("\nTop 15 variables correlated with " "TASA_DESERCION_MPIO:")


print(corr.head(15).to_string(index=False))


# -------------------------------------------------------------------------
# Lag autocorrelation
# -------------------------------------------------------------------------

if "DESERCION_LAG1" in panel.columns:

    lag_data = panel[[target, "DESERCION_LAG1"]].dropna()

    if len(lag_data) > 0:

        lag_corr = lag_data.corr().loc[target, "DESERCION_LAG1"]

        print(f"\nAutocorrelation lag-1 " f"(R): {lag_corr:.4f}")

        print(f"R²: {lag_corr**2:.4f}")


# =============================================================================
# 11e. 2023 TEST DIAGNOSTICS
# =============================================================================

print("\n>>> 2023 TEST SET DIAGNOSTICS")


test_panel = panel[panel["PERIODO_ANIO"] == TEST_YEAR].copy()


print(f"2023 rows: " f"{len(test_panel):,}")


print(f"2023 unique sedes: " f"{test_panel['SEDE_CODIGO'].nunique():,}")


print(f"2023 target coverage: " f"{test_panel[target].notna().mean()*100:.2f}%")


print(f"2023 lag1 coverage: " f"{test_panel['DESERCION_LAG1'].notna().mean()*100:.2f}%")


# =============================================================================
# 12. EXPORT
# =============================================================================

print("\n>>> MODULE 12: EXPORTING RESULTS")


# -------------------------------------------------------------------------
# Parquet
# -------------------------------------------------------------------------

panel.to_parquet(PROCESSED / "panel_maestro.parquet", index=False)


# -------------------------------------------------------------------------
# CSV
# -------------------------------------------------------------------------

panel.to_csv(PROCESSED / "panel_maestro.csv", index=False, encoding="utf-8-sig")


# -------------------------------------------------------------------------
# Excel diagnostics
# -------------------------------------------------------------------------

with pd.ExcelWriter(PROCESSED / "diagnostico_panel.xlsx", engine="openpyxl") as writer:

    # General completeness
    diag.to_excel(writer, sheet_name="Completitud", index=False)

    # Correlations
    corr.to_excel(writer, sheet_name="Correlaciones_target", index=False)

    # Coverage by year
    coverage_by_year.to_excel(writer, sheet_name="Coverage_por_anio", index=False)

    # Train / Test coverage
    coverage_train_test.to_excel(writer, sheet_name="Coverage_train_test", index=False)

    # Descriptive statistics
    panel.describe(include="all").T.reset_index().to_excel(
        writer, sheet_name="Descriptivos", index=False
    )

    # ---------------------------------------------------------------------
    # SIMAT no-match
    # ---------------------------------------------------------------------

    sin_match = (
        panel[panel["TASA_DESERCION_MPIO"].isna()][
            [
                "SEDE_CODIGO",
                "PERIODO_ANIO",
                "COD_MPIO_DANE",
            ]
        ]
        .drop_duplicates()
        .head(500)
    )

    sin_match.to_excel(writer, sheet_name="Sin_match_SIMAT", index=False)

    # ---------------------------------------------------------------------
    # 2023 test set
    # ---------------------------------------------------------------------

    test_panel.head(5000).to_excel(writer, sheet_name="Test_2023_sample", index=False)


# =============================================================================
# 13. FINAL OUTPUT
# =============================================================================

print("\n" + "=" * 70)

print("PIPELINE COMPLETE")

print("=" * 70)


print(f"Panel: " f"{panel.shape[0]:,} rows x " f"{panel.shape[1]} columns")


print(f"Years: " f"{sorted(panel['PERIODO_ANIO'].unique())}")


print(f"TRAIN: " f"{TRAIN_YEARS}")


print(f"TEST: " f"{TEST_YEAR}")


print(f"Unique sedes: " f"{panel['SEDE_CODIGO'].nunique():,}")


print("\nOutputs:")


print(f"  panel_maestro.parquet -> " f"{PROCESSED}")


print(f"  panel_maestro.csv -> " f"{PROCESSED}")


print(f"  diagnostico_panel.xlsx -> " f"{PROCESSED}")


print("\n[OK] PIPELINE COMPLETE")


>>> MODULE 1: LOADING C-600 — SKELETON (Jornadas_nivel)
  Cargando 2018: Jornadas_nivel_2018.csv
  [OK] Jornadas_nivel_2018.csv | enc:cp1252 | sep:',' | 127,310 rows
  [DEBUG] Sedes válidas: 127,310
  [DEBUG] Sedes únicas: 53,202
  Cargando 2019: Jornadas_nivel_2019.csv
  [OK] Jornadas_nivel_2019.csv | enc:utf-8-sig | sep:';' | 130,645 rows
  [DEBUG] Sedes válidas: 130,645
  [DEBUG] Sedes únicas: 53,527
  Cargando 2020: Jornadas_nivel_2020.csv
  [OK] Jornadas_nivel_2020.csv | enc:utf-8-sig | sep:';' | 129,297 rows
  [DEBUG] Sedes válidas: 129,297
  [DEBUG] Sedes únicas: 53,484
  Cargando 2021: Jornadas_nivel_2021.CSV
  [OK] Jornadas_nivel_2021.CSV | enc:cp1252 | sep:',' | 128,432 rows
  [DEBUG] Sedes válidas: 128,432
  [DEBUG] Sedes únicas: 53,066
  Cargando 2022: Jornadas_nivel_2022.CSV
  [OK] Jornadas_nivel_2022.CSV | enc:cp1252 | sep:',' | 129,726 rows
  [DEBUG] Sedes válidas: 129,726
  [DEBUG] Sedes únicas: 53,184
  Cargando 2023: Jornadas_nivel_2023.CSV
  [OK] Jornadas_nivel_2023